# Variant 04. Restaurant Performance Analysis

**Domain:** Restaurant Performance, Kyrgyzstan. All money is in KGS. Data are synthetic.
Tools used: NumPy, Pandas, Matplotlib only. The original CSV files are never modified; all cleaning is done on a copy.

## TASK 1. Initial Data Exploration

Both files start with a BOM character, so they are read with `encoding="utf-8-sig"`. Otherwise the first column would be called `\ufeffTip Amount`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

raw = pd.read_csv("04_restaurant_raw.csv", encoding="utf-8-sig")
lookup = pd.read_csv("04_restaurant_lookup.csv", encoding="utf-8-sig")
print("raw shape:", raw.shape, "| lookup shape:", lookup.shape)

raw shape: (840, 12) | lookup shape: (7, 3)


**1.1 Shape.** The main table has 840 rows and 12 columns; the lookup table has 7 rows and 3 columns.

**1.2 First 10 rows:**

In [2]:
raw.head(10)

,Tip Amount,Payment Method,Customer Count,Waiter,Unit Price,Quantity,dish_name_raw,Dish Category,Meal Type,Order Date,Branch,Order ID
0,No,Basic,No,Premium,19094,38,Yes,Category A,Standard,2025-06-15,Bishkek,04_-00000
1,Yes,Basic,No,Premium,41266,21,Yes,Category B,Premium,28.11.2024,Bishkek,04_-00001
2,No,Basic,No,Premium,48267,16,Yes,Category A,Premium,2024-12-08,Bishkek,04_-00002
3,Yes,Premium,Yes,Basic,25182,49,No,Category C,Basic,2024-02-09,Tokmok,04_-00003
4,No,Basic,Yes,Standard,22174,39,No,Category C,Basic,23.12.2025,Bishkek,04_-00004
5,Yes,Premium,Yes,Premium,30596,41,No,Category A,Basic,16.05.2024,Bishkek,04_-00005
6,Yes,Standard,No,Standard,4950,14,No,Category C,Basic,2025-10-14,Bishkek,04_-00006
7,No,Standard,No,Premium,39254,4,Yes,Category A,Premium,04.04.2025,Jalal-Abad,04_-00007
8,No,Standard,No,Basic,16132,25,Yes,Category A,Basic,2025-09-14,Karakol,04_-00008
9,Yes,Basic,No,Basic,49238,11,Yes,Category C,Basic,27.06.2024,Tokmok,04_-00009


**1.3 Column names and types (`.info()`).** Column names mix spaces, capitals and snake_case (`dish_name_raw`). `Order Date` is `object` (text), not a date. `Unit Price` and `Quantity` are already `int64` in this file, although the task describes `Quantity` as text; the `astype()` conversion is still applied in Task 2 so the notebook is safe if the text version is used.

In [3]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 840 entries, 0 to 839
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Tip Amount      840 non-null    str  
 1   Payment Method  840 non-null    str  
 2   Customer Count  840 non-null    str  
 3   Waiter          840 non-null    str  
 4   Unit Price      840 non-null    int64
 5   Quantity        840 non-null    int64
 6   dish_name_raw   840 non-null    str  
 7   Dish Category   825 non-null    str  
 8   Meal Type       827 non-null    str  
 9   Order Date      840 non-null    str  
 10  Branch          840 non-null    str  
 11  Order ID        840 non-null    str  
dtypes: int64(2), str(10)
memory usage: 78.9 KB


**1.4 Unique values.** `Meal Type` has 11 spellings for 3 real categories (`Basic`, `basic`, ` Basic `, ...) plus missing values. `Payment Method` is clean: only `Basic`, `Premium`, `Standard`.

In [4]:
print(raw["Meal Type"].unique())
print(raw["Meal Type"].value_counts(dropna=False))
print(raw["Payment Method"].unique())
print(raw["Payment Method"].value_counts(dropna=False))

<StringArray>
[  'Standard',    'Premium',      'Basic',    'premium',          nan,
   'standard',    ' Basic ',  ' Premium ', ' Standard ',      'basic',
 ' standard ']
Length: 11, dtype: str
Meal Type
Premium       269
Basic         258
Standard      245
premium        13
NaN            13
standard       10
 Basic          8
 Standard       8
basic           8
 Premium        7
 standard       1
Name: count, dtype: int64
<StringArray>
['Basic', 'Premium', 'Standard']
Length: 3, dtype: str
Payment Method
Standard    292
Premium     275
Basic       273
Name: count, dtype: int64


**1.5 Missing values.** `Meal Type` has 13 missing and `Dish Category` has 15 missing. No other column has missing values (the "N/A" mentioned in the task notes does not occur in this file).

In [5]:
raw.isna().sum()

Tip Amount         0
Payment Method     0
Customer Count     0
Waiter             0
Unit Price         0
Quantity           0
dish_name_raw      0
Dish Category     15
Meal Type         13
Order Date         0
Branch             0
Order ID           0
dtype: int64

**1.6 Duplicates.** There are 19 fully identical rows (the same `Order ID` and all other fields), shown below. There are 20 repeated `Order ID`s, so one `Order ID` is reused by two *different* records. That one is not a copy and is kept.

In [6]:
print("duplicate rows:", raw.duplicated().sum())
print("duplicate Order IDs:", raw["Order ID"].duplicated().sum())
raw[raw.duplicated(keep=False)].sort_values("Order ID").head(6)

duplicate rows: 19
duplicate Order IDs: 20


,Tip Amount,Payment Method,Customer Count,Waiter,Unit Price,Quantity,dish_name_raw,Dish Category,Meal Type,Order Date,Branch,Order ID
28,Yes,Basic,Yes,Premium,19088,13,No,Category B,Premium,12.04.2025,Jalal-Abad,04_-00028
837,Yes,Basic,Yes,Premium,19088,13,No,Category B,Premium,12.04.2025,Jalal-Abad,04_-00028
831,No,Basic,No,Premium,32078,45,No,Category B,Standard,2025-05-31,Jalal-Abad,04_-00228
228,No,Basic,No,Premium,32078,45,No,Category B,Standard,2025-05-31,Jalal-Abad,04_-00228
835,Yes,Standard,No,Premium,4758,22,Yes,Category D,Premium,21.02.2024,Karakol,04_-00236
236,Yes,Standard,No,Premium,4758,22,Yes,Category D,Premium,21.02.2024,Karakol,04_-00236


**1.7 Variable types and data quality problems.**

- **Numerical:** `Unit Price`, `Quantity`.
- **Categorical:** `Order ID`, `Order Date` (text), `Branch`, `Meal Type`, `Dish Category`, `Payment Method`, `Waiter`, plus `Tip Amount`, `Customer Count`, `dish_name_raw`.

Problems found:

1. `Meal Type` inconsistent: `premium`, ` Basic `, ` standard ` (case and spaces). 13 values missing.
2. `Dish Category` has 15 missing values.
3. 19 exact duplicate rows.
4. `Unit Price` has extreme outliers: the max is 1,728,760 while the median is 26,475 (about 65 times larger).
5. `Order Date` is text in two formats: `2025-06-15` and `28.11.2024` (415 of 840 rows use dots).
6. Column names are inconvenient (spaces, capitals, mixed style) and the order is illogical (the ID is last).
7. **Misleading columns:** `Tip Amount`, `Customer Count` and `dish_name_raw` contain only `Yes`/`No`. They are not amounts or counts. `Payment Method` and `Waiter` contain `Basic/Premium/Standard`, not real payment types or names. This cannot be fixed; it affects Task 3.3 (see there).
8. The lookup table has duplicate keys that differ only by case (`Basic` and `basic`, with different segments) and one row with an empty key.

## TASK 2. Data Cleaning and Preparation

All cleaning is done on the copy `df`; `raw` stays untouched.

**2.1 Missing values.** I cannot know the real `Meal Type` or `Dish Category` of these orders. Guessing (mode fill) would invent data and would add to the biggest category. Dropping rows would throw away valid prices and quantities. So the missing values are labelled `Unknown`: the rows stay, and the gap stays visible.

In [7]:
df = raw.copy()
df["Meal Type"] = df["Meal Type"].fillna("Unknown")
df["Dish Category"] = df["Dish Category"].fillna("Unknown")
print(df[["Meal Type", "Dish Category"]].isna().sum().to_dict())
print((df["Meal Type"] == "Unknown").sum(), (df["Dish Category"] == "Unknown").sum())

{'Meal Type': 0, 'Dish Category': 0}
13 15


**2.2 Standardise `Meal Type`.** `.str.strip()` removes spaces and `.str.title()` unifies case. Mapping: `basic`, ` Basic ` → `Basic`; `premium`, ` Premium ` → `Premium`; `standard`, ` standard `, ` Standard ` → `Standard`. Result: 3 categories + `Unknown`.

In [8]:
df["Meal Type"] = df["Meal Type"].str.strip().str.title()
mapping = {"Basic": "basic / ' Basic '", "Premium": "premium / ' Premium '", "Standard": "standard / ' standard ' / ' Standard '"}
print(df["Meal Type"].value_counts())

Meal Type
Premium     289
Basic       274
Standard    264
Unknown      13
Name: count, dtype: int64


**2.3 Duplicates.** `drop_duplicates()` is run *after* standardising so that rows that differ only by spelling are also caught. It removed 19 rows (840 → 821). One `Order ID` is still repeated, but with different content, so it is a different record and stays.

In [9]:
before = len(df)
df = df.drop_duplicates()
print("removed:", before - len(df), "| rows now:", len(df))
print("Order IDs still repeated:", df["Order ID"].duplicated().sum())

removed: 19 | rows now: 821
Order IDs still repeated: 1


**2.4 Outliers in `Unit Price`.** I use the IQR rule (covered in class): a value is an outlier if it is below Q1 − 1.5·IQR or above Q3 + 1.5·IQR. Q1 = 13,191 and Q3 = 38,396, so the upper bound is 76,203.5. **10 values** are above it (107,600 up to 1,728,760); the lower bound is negative, so no low outliers are flagged. All other prices are below 50,000, so the 10 are clearly separated from the data. I remove them instead of capping, because I cannot know the real price (it looks like an entry error, e.g. extra digits) and a capped value would be invented.

In [10]:
q1, q3 = df["Unit Price"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
mask = (df["Unit Price"] < low) | (df["Unit Price"] > high)
print(f"Q1={q1}, Q3={q3}, IQR={iqr}, bounds=({low}, {high})")
print("outliers:", mask.sum())
df.loc[mask, ["Order ID", "Unit Price", "Quantity"]].sort_values("Unit Price")

Q1=13191.0, Q3=38396.0, IQR=25205.0, bounds=(-24616.5, 76203.5)
outliers: 10


,Order ID,Unit Price,Quantity
559,04_-00559,107600,39
545,04_-00545,364360,12
11,04_-00011,1114840,27
638,04_-00638,1257240,39
709,04_-00709,1317680,3
637,04_-00637,1424200,2
292,04_-00292,1508520,28
718,04_-00718,1520640,6
331,04_-00331,1635440,42
485,04_-00485,1728760,36


In [11]:
df = df[~mask].copy()
print("rows after removing outliers:", len(df), "| max price:", df["Unit Price"].max())

rows after removing outliers: 811 | max price: 49887


**2.5 `Quantity` to numeric** with `astype(int)`.

In [12]:
df["Quantity"] = df["Quantity"].astype(int)
print(df["Quantity"].dtype)

int64


**2.6 `Order Date` to datetime.** `pd.to_datetime` on a mixed column would guess the format and could swap day and month (`12.04.2025` is 12 April, not 4 December). So I split by format: values with `-` are `YYYY-MM-DD`, the others are `DD.MM.YYYY`. Each part is parsed with an explicit `format=`, then they are joined back by index. The range is 2024-01-01 to 2025-12-31.

In [13]:
iso = df["Order Date"].str.contains("-")
d1 = pd.to_datetime(df.loc[iso, "Order Date"], format="%Y-%m-%d")
d2 = pd.to_datetime(df.loc[~iso, "Order Date"], format="%d.%m.%Y")
df["Order Date"] = pd.concat([d1, d2]).sort_index()
print(df["Order Date"].dtype, df["Order Date"].min(), df["Order Date"].max())

datetime64[us] 2024-01-01 00:00:00 2025-12-31 00:00:00


**2.7 Column names and order.** Names are lowercase snake_case. `Unit Price` becomes `unit_price`. The order is logical: identifiers and time, then place, then product, then payment/people, then numbers.

In [14]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df = df[["order_id", "order_date", "branch", "meal_type", "dish_category", "dish_name_raw",
         "payment_method", "waiter", "customer_count", "quantity", "unit_price", "tip_amount"]]
df.columns.tolist()

['order_id',
 'order_date',
 'branch',
 'meal_type',
 'dish_category',
 'dish_name_raw',
 'payment_method',
 'waiter',
 'customer_count',
 'quantity',
 'unit_price',
 'tip_amount']

**2.8 Merge with the lookup.** The lookup has the same case problem as the main table: `Basic` and `basic` have different segments, and one row has an empty key. If I merged it as is, orders would be duplicated or wrongly matched. I keep only the properly spelled keys (`Basic`, `Premium`, `Standard`) and drop the lowercase duplicates and the empty-key row. I cannot tell which of the two conflicting segments is correct; the properly spelled key is the choice I can justify.

In [15]:
lk = lookup.copy()
lk.columns = lk.columns.str.strip().str.lower().str.replace(" ", "_")
lk = lk.dropna(subset=["meal_type"])
lk = lk[lk["meal_type"] == lk["meal_type"].str.title()]
print(lk)

  meal_type business_segment priority
0     Basic        Segment 1     High
1   Premium        Segment 2   Medium
2  Standard        Segment 3      Low


I use a **left merge**: every main record must stay. The `assert` confirms the row count did not change. **Unmatched: 12 rows**, all with `meal_type = Unknown`, which has no lookup entry; their `business_segment` and `priority` remain empty.

In [16]:
n = len(df)
df = df.merge(lk, on="meal_type", how="left")
assert len(df) == n
print("unmatched rows:", df["business_segment"].isna().sum())
print(df.loc[df["business_segment"].isna(), "meal_type"].value_counts())

unmatched rows: 12
meal_type
Unknown    12
Name: count, dtype: int64


**2.9 Final check and save.** The only remaining missing values are those 12 unmatched lookup fields. Types are correct. 811 rows and 14 columns are saved to `cleaned_data.csv` with `index=False`.

In [17]:
print(df.isna().sum()[lambda s: s > 0])
print(df.dtypes)
df.to_csv("cleaned_data.csv", index=False)
print(df.shape)

business_segment    12
priority            12
dtype: int64
order_id                       str
order_date          datetime64[us]
branch                         str
meal_type                      str
dish_category                  str
dish_name_raw                  str
payment_method                 str
waiter                         str
customer_count                 str
quantity                     int64
unit_price                   int64
tip_amount                     str
business_segment               str
priority                       str
dtype: object
(811, 14)
